In [ ]:
import random
import pandas as pd
import numpy as np

from src.utils import load_env, set_seed,load_json, get_logger, get_token
from case_study.utils import set_matplot_style
from eval.utils import format_lcc_llm_res_df_for_analysis, thresholds_f1_table
from src.experiment_config import ExperimentConfig
from src.data_loading import DatasetLoader, get_llm_metaphor_binary_df
from src.db_config import DocAnnotation, Document

from sqlalchemy import create_engine, select, func
set_matplot_style()

env_vars = load_env()
seed = env_vars["RANDOM_SEED"]
set_seed(seed)

logger = get_logger("eval")

In [ ]:
# LOAD CONFIG FOR DATASET OF INTEREST USING ROB RESULTS CONFIG
rob_4c_path = f"{env_vars["RESULTS_DIR"]}/get_candidate_metaphors/rule_based/lcc-large_labeled_metaphor_paths.json"
rb_results = load_json(rob_4c_path)

# load experiment config, update paths
config = ExperimentConfig.from_dict(rb_results["config"], logger=get_logger("eval"), env_vars=env_vars)

# get db session for lcc dataset
config.skip_load = True
data_loader = DatasetLoader(config)
lcc_session = data_loader.load_preprocessed_data()

In [ ]:
qwen_binary_lcc_llm_res_path = f"{env_vars["RESULTS_DIR"]}/metaphor_classification/llm/lcc_source_verb_target_noun_binary_met_class_qwen.json"
qwen_binary_lcc_llm_df = get_llm_metaphor_binary_df(qwen_binary_lcc_llm_res_path, logger=logger)

In [ ]:
# check llm binary classifications
qwen_binary_lcc_llm_df.groupby("llm_met_class").count()

In [ ]:
# DATASET STATS
# print a database document example
stmt = select(Document)
res = lcc_session.execute(stmt)
for row in res:
    row = row[0]
    print(row.text)
    break

num_sents = [row[0].num_sentences for row in res]
total_num_sents = sum(num_sents)
av_num_sents = total_num_sents / len(num_sents)

# get the counts 
full_count = lcc_session.scalar(select(func.count()).select_from(Document))
print(f"\n{full_count} documents with {total_num_sents} sentences in database with an average of {round(av_num_sents, 3)} sentences each")

ann_documents = lcc_session.query(Document).join(Document.doc_annotations).filter_by(annotation_type='human_metaphor_score').all()
ann_num_sents = [doc.num_sentences for doc in ann_documents]
ann_total_sentences = sum(ann_num_sents)
ann_av_sentences = ann_total_sentences / len(ann_num_sents)
print(f"{len(ann_documents)} documents with {ann_total_sentences} sentences are annotated (average sentences = {round(ann_av_sentences, 3)})")

In [ ]:
print("\nQWEN BINARY STATS")
print(f"{len(qwen_binary_lcc_llm_df)} LLM sdp-level annotations for metaphor classification")
qwen_binary_lcc_metaphor_df = qwen_binary_lcc_llm_df[qwen_binary_lcc_llm_df["llm_met_class"] == True]
print(f"{len(qwen_binary_lcc_metaphor_df)} are metaphorical")

In [ ]:
# match results to hand annotations
qwen_binary_analysis_df = format_lcc_llm_res_df_for_analysis(lcc_session, qwen_binary_lcc_llm_df)

In [ ]:
print("\n BINARY QWEN STATS")
print(f"{len(qwen_binary_analysis_df)} LLM sdp-level annotations for metaphor score (matched to document score with matching source and target word)")
qwen_binary_lcc_metaphor_df = qwen_binary_analysis_df[qwen_binary_analysis_df["llm_met_class"] == True]
print(f"{len(qwen_binary_lcc_metaphor_df)} are classified metaphorical")

#  BINARY QWEN STATS
# 10211 LLM sdp-level annotations for metaphor score (matched to document score with matching source and target word)
# 6347 are classified metaphorical

In [ ]:
qwen_binary_analysis_df["human_metaphor_score"] = qwen_binary_analysis_df["human_met_score"]
f1_table = thresholds_f1_table(qwen_binary_analysis_df, "llm_met_class", "", [1, 2, 3])

print("Binary Qwen Results: ")
f1_table

In [ ]:
# random baseline 
qwen_binary_analysis_df["random"] = qwen_binary_analysis_df["human_metaphor_score"].apply(lambda x: random.choice([0, 1]))
random_f1_table = thresholds_f1_table(qwen_binary_analysis_df, "random", "", [1, 2, 3])

print("Random Results: ")
random_f1_table